# Reward Surface Plots

Two surface plots showing total episode reward as the sum of:
1. **Success**: cumulative progress signal + success bonus
2. **Death**: cumulative progress signal − death penalty

In [13]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Hyperparameters (adjust these) ──
lambda_p    = 0.04
lambda_rho  = 0.4
lambda_s    = 0.001
lambda_t    = 40.0
lambda_d    = 1.0
reach_bonus = 150.0
max_steps   = 1000

In [14]:
# ── Axis ranges ──
progress_max = lambda_p * max_steps
success_max  = reach_bonus + lambda_t
death_max    = lambda_d * reach_bonus

N = 100
progress = np.linspace(-progress_max, progress_max, N)
success  = np.linspace(0, success_max, N)
death    = np.linspace(0, death_max, N)

P_s, S = np.meshgrid(progress, success)
P_d, D = np.meshgrid(progress, death)

Z_success = P_s + S
Z_death   = P_d - D

In [15]:
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "surface"}, {"type": "surface"}]],
    subplot_titles=["Success: progress + bonus", "Death: progress − penalty"],
    horizontal_spacing=0.05,
)

fig.add_trace(
    go.Surface(
        x=P_s, y=S, z=Z_success,
        colorscale="Greens", showscale=False,
        hovertemplate="progress: %{x:.0f}<br>success bonus: %{y:.0f}<br>total: %{z:.0f}<extra></extra>",
    ),
    row=1, col=1,
)

fig.add_trace(
    go.Surface(
        x=P_d, y=D, z=Z_death,
        colorscale="Reds_r", showscale=False,
        hovertemplate="progress: %{x:.0f}<br>death penalty: %{y:.0f}<br>total: %{z:.0f}<extra></extra>",
    ),
    row=1, col=2,
)

fig.update_layout(
    height=550, width=1100,
    title_text=f"λ_p={lambda_p}  λ_t={lambda_t}  λ_d={lambda_d}  reach={reach_bonus}",
    margin=dict(l=0, r=0, t=60, b=0),
)

fig.update_scenes(
    dict(
        xaxis_title="Cumul. progress",
        yaxis_title="Success bonus",
        zaxis_title="Total reward",
    ),
    row=1, col=1,
)
fig.update_scenes(
    dict(
        xaxis_title="Cumul. progress",
        yaxis_title="Death penalty",
        zaxis_title="Total reward",
    ),
    row=1, col=2,
)

fig.show()

## Success vs non-success dominance check

The progress signal telescopes: `Σ λ_p(J_{t-1} − J_t) = λ_p(J_0 − J_T)`.

- **Non-success** (timeout): `J_T > 0`, so cumul. progress ∈ `[−λ_p J_0, +λ_p J_0)`. Runs full `max_steps`.
- **Success**: `J_T = 0`, so cumul. progress `= λ_p J_0 > 0`. Runs ≤ `max_steps`.

For the check to hold, the **best non-success** episode (max progress, min risk, fewest steps) must yield less reward than the **worst success** episode (min bonus `time_ratio=0`, max risk, most steps).

Axes: cumul. progress × cumul. risk penalty.
Green = worst-case success. Red = non-success. Green must be entirely above red.

In [16]:
# ── Plausible per-episode ranges (adjust these) ──
# Progress telescopes to λ_p * (J_0 - J_T).
# J_0 (cost-to-go at spawn) is roughly proportional to distance × avg terrain cost.
# On a 250×250 map with ~200-cell journeys and avg cost ~2.5: J_0 ≈ 500
J0_max = 500.0  # max plausible initial cost-to-go

# Non-success: J_T > 0, so progress < λ_p * J_0. Agent could also wander backward.
nonsuccess_progress_max = lambda_p * J0_max      # ≈ 25, approached target but didn't reach
nonsuccess_progress_min = -lambda_p * J0_max     # ≈ -25, wandered away

# Success: J_T = 0, progress = λ_p * J_0. Range depends on J_0 of the episode.
success_progress_min = lambda_p * 50.0   # short easy episode
success_progress_max = lambda_p * J0_max # long hard episode

# Cumulative risk: Σ λ_ρ ρ_t over the episode.
# ρ_t = drain / (res + 0.5*hp). Typical avg ρ ≈ 0.02-0.1.
# Non-success runs full max_steps; success may be shorter.
avg_rho_min = 0.0
avg_rho_max = 0.15  # aggressive terrain, low resources
risk_cumul_max = lambda_rho * avg_rho_max * max_steps  # worst case: 75

N = 80

# ── Non-success surface (red) ──
# Runs full max_steps, no terminal bonus
ns_progress = np.linspace(nonsuccess_progress_min, nonsuccess_progress_max, N)
ns_risk     = np.linspace(0, risk_cumul_max, N)
NS_P, NS_R  = np.meshgrid(ns_progress, ns_risk)
Z_nonsuccess = NS_P - NS_R - lambda_s * max_steps

# ── Success surface (green, worst case: time_ratio=0) ──
# Gets at least reach_bonus. Progress is always ≥ 0 (reached target).
s_progress = np.linspace(success_progress_min, success_progress_max, N)
s_risk     = np.linspace(0, risk_cumul_max, N)
S_P, S_R   = np.meshgrid(s_progress, s_risk)
Z_success_worst = S_P - S_R - lambda_s * max_steps + reach_bonus  # time_ratio=0

# ── Check ──
best_nonsuccess = Z_nonsuccess.max()
worst_success   = Z_success_worst.min()
gap = worst_success - best_nonsuccess

fig2 = go.Figure()

fig2.add_trace(go.Surface(
    x=NS_P, y=NS_R, z=Z_nonsuccess,
    colorscale=[[0, "rgba(200,0,0,0.8)"], [1, "rgba(255,120,120,0.8)"]],
    showscale=False, name="Non-success (timeout)",
    hovertemplate="progress: %{x:.1f}<br>risk: %{y:.1f}<br>reward: %{z:.1f}<extra>non-success</extra>",
))

fig2.add_trace(go.Surface(
    x=S_P, y=S_R, z=Z_success_worst,
    colorscale=[[0, "rgba(0,140,0,0.8)"], [1, "rgba(0,255,0,0.8)"]],
    showscale=False, name="Success (worst: time_ratio=0)",
    hovertemplate="progress: %{x:.1f}<br>risk: %{y:.1f}<br>reward: %{z:.1f}<extra>success</extra>",
))

fig2.update_layout(
    height=650, width=900,
    title_text=(
        f"Best non-success = {best_nonsuccess:.1f}  |  Worst success = {worst_success:.1f}  |  "
        f"Gap = {gap:.1f}  {'✓' if gap > 0 else '✗ OVERLAP'}"
    ),
    scene=dict(
        xaxis_title="Cumul. progress signal",
        yaxis_title="Cumul. risk penalty",
        zaxis_title="Total episode reward",
    ),
    margin=dict(l=0, r=0, t=60, b=0),
)

fig2.show()
print(f"Best non-success:  progress={nonsuccess_progress_max:.1f}, risk=0  →  {best_nonsuccess:.1f}")
print(f"Worst success:     progress={success_progress_min:.1f}, risk={risk_cumul_max:.1f}  →  {worst_success:.1f}")
print(f"Gap = {gap:.1f}  →  {'Success always dominates ✓' if gap > 0 else 'VIOLATED — surfaces overlap ✗'}")

Best non-success:  progress=20.0, risk=0  →  19.0
Worst success:     progress=2.0, risk=60.0  →  91.0
Gap = 72.0  →  Success always dominates ✓
